# 5024CMD – Loan data: sampling, categorisation, ARFF and association rules
Inputs: `fico`, `int.rate`, `dti`, `revol.util`  |  Output: `not.fully.paid`

If `mlxtend` is missing, run `pip install mlxtend` in the Anaconda prompt. Keep `loan_data.csv` in the same folder as this notebook.

## Task 1 – Extract 50 records

In [1]:
import pandas as pd
df = pd.read_csv('loan_data.csv')
cols = ['fico','int.rate','dti','revol.util','not.fully.paid']
sample = df.sample(50, random_state=42)[cols].reset_index(drop=True)   # fixed seed = repeatable
print(sample.shape); sample.head()

(50, 5)


,fico,int.rate,dti,revol.util,not.fully.paid
0,657,0.1545,14.52,82.2,0
1,732,0.1183,10.89,40.9,0
2,692,0.1146,13.73,85.8,1
3,672,0.1646,28.16,25.8,0
4,702,0.1221,3.99,7.2,0


## Task 2 – Numeric to categorical (equal-width, 3 levels, same method as the Excel example)
Interval = (Max − Min) / 3. Value < Min+Interval → Low; < Min+2×Interval → Medium; otherwise High.

In [2]:
num = ['fico','int.rate','dti','revol.util']
cat = pd.DataFrame()
scale = {}
for c in num:
    mn, mx = sample[c].min(), sample[c].max()
    iv = (mx - mn) / 3
    scale[c] = dict(min=mn, max=mx, interval=iv, low_below=mn+iv, medium_below=mn+2*iv)
    cat[c] = pd.cut(sample[c], bins=[-float('inf'), mn+iv, mn+2*iv, float('inf')],
                    labels=['Low','Medium','High'], right=False)
cat['not.fully.paid'] = sample['not.fully.paid'].map({0:'Paid', 1:'Not_Paid'})
display(pd.DataFrame(scale).T.round(4))
cat.head()

,min,max,interval,low_below,medium_below
fico,642.00,777.000,45.0000,687.0000,732.0000
int.rate,0.08,0.167,0.0290,0.1090,0.1380
dti,0.30,28.160,9.2867,9.5867,18.8733
revol.util,6.20,104.200,32.6667,38.8667,71.5333


,fico,int.rate,dti,revol.util,not.fully.paid
0,Low,High,Medium,High,Paid
1,High,Medium,Medium,Medium,Paid
2,Medium,Medium,Medium,High,Not_Paid
3,Low,High,High,Low,Paid
4,Medium,Medium,Low,Low,Paid


In [3]:
cat['not.fully.paid'].value_counts()

not.fully.paid
Paid        44
Not_Paid     6
Name: count, dtype: int64

## Task 3 – CSV to ARFF (WEKA)

In [4]:
def to_arff(frame, relation, path):
    lines = [f'@relation {relation}', '']
    for c in num:
        lines.append(f"@attribute {c.replace('.','_')} {{Low,Medium,High}}")
    lines += ['@attribute not_fully_paid {Paid,Not_Paid}', '', '@data']
    lines += [','.join(map(str, r)) for r in frame.itertuples(index=False)]
    open(path, 'w').write('\n'.join(lines) + '\n')
to_arff(cat, 'loan_50', 'loan_50.arff')
print(open('loan_50.arff').read()[:450])

@relation loan_50

@attribute fico {Low,Medium,High}
@attribute int_rate {Low,Medium,High}
@attribute dti {Low,Medium,High}
@attribute revol_util {Low,Medium,High}
@attribute not_fully_paid {Paid,Not_Paid}

@data
Low,High,Medium,High,Paid
High,Medium,Medium,Medium,Paid
Medium,Medium,Medium,High,Not_Paid
Low,High,High,Low,Paid
Medium,Medium,Low,Low,Paid
Medium,Medium,High,Medium,Paid
High,Low,Medium,Low,Paid
Medium,Low,Medium,Low,Paid
Medium,Mediu


## Task 4 – Association rules (Apriori)
Two runs are made, matching Section 7 of the report:

* **Run A** – the 10-record subset (`loan_10`): minimum support 0.3, minimum confidence 0.8, lift > 1. Rules may involve any attribute. This reproduces the WEKA run.
* **Run B** – all 50 records (`loan_50`): class-association check for rules ending in `Not_Paid` (minimum support 0.05, minimum confidence 0.5).

The 10 records are 7 Paid + 3 Not_Paid, chosen on purpose so that the target value appears often enough to form rules. State this choice in your report. Because the subset is 70 % Paid, any rule with confidence 1.00 ending in Paid has a lift of 1 / 0.70 = 1.43, which is the maximum possible.

In [5]:
sub = pd.concat([cat[cat['not.fully.paid']=='Paid'].sample(7, random_state=1),
                 cat[cat['not.fully.paid']=='Not_Paid'].sample(3, random_state=1)]).sort_index()
to_arff(sub, 'loan_10', 'loan_10.arff')
sub

,fico,int.rate,dti,revol.util,not.fully.paid
3,Low,High,High,Low,Paid
4,Medium,Medium,Low,Low,Paid
10,Low,High,Medium,Medium,Not_Paid
15,Low,High,Low,High,Not_Paid
22,Medium,Medium,High,High,Paid
24,Medium,Medium,High,High,Paid
25,Medium,Medium,Medium,Low,Paid
34,Low,High,Medium,High,Paid
37,High,Medium,Medium,Low,Not_Paid
47,Medium,Medium,Low,Low,Paid


In [6]:
# Run A: 10-record subset, min support 0.3, min confidence 0.8, lift > 1
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

pd.set_option('display.max_colwidth', None)   # show full rule text (no "..." truncation)

tx = [[f'{c}={v}' for c, v in row.items()] for _, row in sub.iterrows()]
te = TransactionEncoder()
onehot = pd.DataFrame(te.fit(tx).transform(tx), columns=te.columns_)
freq = apriori(onehot, min_support=0.3, use_colnames=True)
print('Frequent itemsets:', len(freq))

rules = association_rules(freq, metric='confidence', min_threshold=0.8)
rules = rules[rules.lift > 1].copy()
rules['rule'] = (rules.antecedents.map(lambda s: ' & '.join(sorted(s)))
                 + '  =>  '
                 + rules.consequents.map(lambda s: ' & '.join(sorted(s))))
print('Rules (lift > 1):', len(rules))

# Keep only rules whose right-hand side is the outcome (single item: Paid or Not_Paid)
target = rules[rules.consequents.map(
    lambda s: len(s) == 1 and list(s)[0].startswith('not.fully.paid'))]
print('Rules ending in the outcome:', len(target))

target.sort_values(['confidence', 'lift', 'support'], ascending=False)[
    ['rule', 'support', 'confidence', 'lift']].round(2)

Frequent itemsets: 25
Rules (lift > 1): 20
Rules ending in the outcome: 7


,rule,support,confidence,lift
5,fico=Medium => not.fully.paid=Paid,0.5,1.00,1.43
10,fico=Medium & int.rate=Medium => not.fully.paid=Paid,0.5,1.00,1.43
0,dti=High => not.fully.paid=Paid,0.3,1.00,1.43
15,fico=Medium & revol.util=Low => not.fully.paid=Paid,0.3,1.00,1.43
17,fico=Medium & int.rate=Medium & revol.util=Low => not.fully.paid=Paid,0.3,1.00,1.43
6,int.rate=Medium => not.fully.paid=Paid,0.5,0.83,1.19
8,revol.util=Low => not.fully.paid=Paid,0.4,0.80,1.14


In [7]:
# All 20 rules, for comparison with the WEKA output (WEKA needs numRules >= 30 to list them all)
rules.sort_values(['confidence', 'lift', 'support'], ascending=False)[
    ['rule', 'support', 'confidence', 'lift']].round(2)

,rule,support,confidence,lift
1,int.rate=High => fico=Low,0.4,1.00,2.50
2,fico=Low => int.rate=High,0.4,1.00,2.50
11,int.rate=Medium & not.fully.paid=Paid => fico=Medium,0.5,1.00,2.00
12,fico=Medium => int.rate=Medium & not.fully.paid=Paid,0.5,1.00,2.00
18,int.rate=Medium & not.fully.paid=Paid & revol.util=Low => fico=Medium,0.3,1.00,2.00
19,fico=Medium & revol.util=Low => int.rate=Medium & not.fully.paid=Paid,0.3,1.00,2.00
3,fico=Medium => int.rate=Medium,0.5,1.00,1.67
9,fico=Medium & not.fully.paid=Paid => int.rate=Medium,0.5,1.00,1.67
14,fico=Medium & revol.util=Low => int.rate=Medium,0.3,1.00,1.67
16,fico=Medium & not.fully.paid=Paid & revol.util=Low => int.rate=Medium,0.3,1.00,1.67


**Reading the table:** support = share of the 10 records containing the whole rule; confidence = how often the right side holds when the left side does; lift > 1 = the items occur together more often than chance.

* Only rules whose right-hand side is the outcome (`Paid` or `Not_Paid`) can drive a prediction. Rules such as `fico=Low => int.rate=High` describe how two inputs occur together.
* All rules ending in the outcome end in `Paid`. No rule ends in `Not_Paid`, because the three Not_Paid loans share no value on any attribute.
* The lift of 1.43 is a ceiling set by the 70/30 subset design, not evidence of a strong association. With only 10 records the rules illustrate the method rather than prove anything.

### Run B – class-association check for `Not_Paid` (all 50 records)
Only 6 of the 50 records are Not_Paid, so the minimum support is lowered to 0.05 (at least 3 loans) and the minimum confidence to 0.5. If the table below is empty, no rule ending in `Not_Paid` reached these thresholds.

In [13]:
# Run B: all 50 records, looking only for rules that end in Not_Paid
tx50 = [[f'{c}={v}' for c, v in row.items()] for _, row in cat.iterrows()]
te50 = TransactionEncoder()
onehot50 = pd.DataFrame(te50.fit(tx50).transform(tx50), columns=te50.columns_)
freq50 = apriori(onehot50, min_support=0.05, use_colnames=True)
rules50 = association_rules(freq50, metric='confidence', min_threshold=0.5)
rules50['rule'] = (rules50.antecedents.map(lambda s: ' & '.join(sorted(s)))
                   + '  =>  '
                   + rules50.consequents.map(lambda s: ' & '.join(sorted(s))))

notpaid = rules50[rules50.consequents.map(lambda s: s == frozenset({'not.fully.paid=Not_Paid'}))]
print('Rules ending in Not_Paid (min support 0.05, min confidence 0.5):', len(notpaid))
notpaid[['rule', 'support', 'confidence', 'lift']].round(2)

Rules ending in Not_Paid (min support 0.05, min confidence 0.5): 0


,rule,support,confidence,lift


In [17]:
path = 'run b weka results'      # change to the new file's exact name
lines = open(path, encoding='utf-8', errors='ignore').read().splitlines()

scheme = next(l for l in lines if l.startswith('Scheme'))
rule_lines = [l for l in lines if re.match(r'^\s*\d+\.\s', l)]
lhs = [l.split('==>')[0] for l in rule_lines]
rhs = [l.split('==>')[1] for l in rule_lines]

print(scheme)
print('Class-association (-A) on:', ' -A' in scheme)
print(*[l for l in lines if l.startswith(('Relation', 'Instances', 'Minimum support', 'Minimum metric'))], sep='\n')
print('\nTotal rules:', len(rule_lines))
print('Rules with Paid on the right:    ', sum('not_fully_paid=Paid' in r for r in rhs))
print('Rules with Not_Paid on the right:', sum('not_fully_paid=Not_Paid' in r for r in rhs))
print('Rules with Not_Paid on the left: ', sum('not_fully_paid=Not_Paid' in x for x in lhs))
print()
print(*[l for l, r in zip(rule_lines, rhs) if 'not_fully_paid=Not_Paid' in r], sep='\n')

Scheme:       weka.associations.Apriori -N 10000 -T 0 -C 0.5 -D 0.05 -U 1.0 -M 0.05 -S -1.0 -A -c -1
Class-association (-A) on: True
Relation:     loan_50
Instances:    50
Minimum support: 0.05 (3 instances)
Minimum metric <confidence>: 0.5

Total rules: 85
Rules with Paid on the right:     85
Rules with Not_Paid on the right: 0
Rules with Not_Paid on the left:  0




### Full-data check of the rules (reproduces Table 6)
Each rule found on the 10 records is compared with the same band in all 9,578 loans. Bands use equal-width cut-offs calculated from the full data, so they differ slightly from the 50-record cut-offs in Task 2. The check is therefore directional, not exact. Run this after Task 1, because it uses `df`.

In [ ]:
# Paid / default rate by Low-Medium-High band, full dataset
num = ['fico', 'int.rate', 'dti', 'revol.util']
base_paid = (df['not.fully.paid'] == 0).mean()
print(f'Overall Paid rate: {base_paid:.1%}   (maximum possible lift of a Paid rule = {1/base_paid:.2f})')

band_rows = []
for c in num:
    mn, mx = df[c].min(), df[c].max()
    iv = (mx - mn) / 3
    band = pd.cut(df[c], bins=[-float('inf'), mn + iv, mn + 2*iv, float('inf')],
                  labels=['Low', 'Medium', 'High'], right=False)
    g = df.groupby(band, observed=True)['not.fully.paid'].agg(n='size', default_rate='mean')
    for lvl, r in g.iterrows():
        band_rows.append({'attribute': c, 'cut-offs': f'{mn+iv:.4g} / {mn+2*iv:.4g}', 'band': lvl,
                          'n': int(r.n), 'default %': round(100*r.default_rate, 1),
                          'paid %': round(100*(1 - r.default_rate), 1)})
bands = pd.DataFrame(band_rows)
bands

In [ ]:
# Check each single-condition Paid rule from Run A against the full data
check = []
for attr, lvl in [('fico', 'Medium'), ('dti', 'High'), ('int.rate', 'Medium'), ('revol.util', 'Low')]:
    r = bands[(bands.attribute == attr) & (bands.band == lvl)].iloc[0]
    paid = r['paid %'] / 100
    check.append({'rule': f'{attr}={lvl} => Paid', 'loans in band': r['n'],
                  'paid % (full data)': r['paid %'], 'full-data lift': round(paid / base_paid, 2),
                  'assessment': 'Consistent' if paid > base_paid else 'Contradicted'})
pd.DataFrame(check)

**Reading the check:** a rule is *consistent* if its band's Paid rate is above the overall 84.0 %, and *contradicted* if it is below. `fico=Medium` and `revol.util=Low` are consistent (`fico=Medium` only marginally, at about 1.01 lift). `dti=High` and `int.rate=Medium` are contradicted, so they are artefacts of the 10-record subset and are not used for prediction.

## Task 5 – Store the data in a database (SQLite)
SQLite is built into Python (`sqlite3`), so no extra installation is needed. The database is saved as `loan_data.db` in the same folder as this notebook. Two tables are created: `loan_sample` (the 50 categorised records) and `loan_full` (all 9,578 numerical records). Run this after Tasks 1 and 2, because it uses `df` and `cat`.

In [9]:
import sqlite3

conn = sqlite3.connect('loan_data.db')   # creates the file if it does not exist
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS loan_sample")
cur.execute("DROP TABLE IF EXISTS loan_full")

cur.execute("""
CREATE TABLE loan_sample (
  loan_id INTEGER PRIMARY KEY,
  fico TEXT CHECK (fico IN ('Low','Medium','High')),
  int_rate TEXT CHECK (int_rate IN ('Low','Medium','High')),
  dti TEXT CHECK (dti IN ('Low','Medium','High')),
  revol_util TEXT CHECK (revol_util IN ('Low','Medium','High')),
  not_fully_paid TEXT CHECK (not_fully_paid IN ('Paid','Not_Paid'))
)""")

cur.execute("""
CREATE TABLE loan_full (
  loan_id INTEGER PRIMARY KEY,
  fico INTEGER, int_rate REAL, dti REAL, revol_util REAL,
  not_fully_paid INTEGER CHECK (not_fully_paid IN (0,1))
)""")
conn.commit()

### Load the records
SQL column names cannot contain dots, so `int.rate` becomes `int_rate` and so on. `pd.cut` returns a categorical type, so the categories are converted to text before writing.

In [10]:
rename = {'int.rate':'int_rate', 'revol.util':'revol_util', 'not.fully.paid':'not_fully_paid'}

# 50 categorised records (from 'cat', Task 2)
s = cat.rename(columns=rename).astype(str)
s.insert(0, 'loan_id', range(1, len(s)+1))
s.to_sql('loan_sample', conn, if_exists='append', index=False)

# 9,578 numerical records (from 'df', Task 1)
f = df[cols].rename(columns=rename)
f.insert(0, 'loan_id', range(1, len(f)+1))
f.to_sql('loan_full', conn, if_exists='append', index=False)
conn.commit()
s.head()

,loan_id,fico,int_rate,dti,revol_util,not_fully_paid
0,1,Low,High,Medium,High,Paid
1,2,High,Medium,Medium,Medium,Paid
2,3,Medium,Medium,Medium,High,Not_Paid
3,4,Low,High,High,Low,Paid
4,5,Medium,Medium,Low,Low,Paid


### Query the database (screenshot these outputs for the appendix)

In [11]:
print(pd.read_sql("SELECT COUNT(*) AS rows_loan_sample FROM loan_sample", conn))
print(pd.read_sql("SELECT COUNT(*) AS rows_loan_full FROM loan_full", conn))
pd.read_sql("SELECT * FROM loan_sample LIMIT 10", conn)

   rows_loan_sample
0                50
   rows_loan_full
0            9578


,loan_id,fico,int_rate,dti,revol_util,not_fully_paid
0,1,Low,High,Medium,High,Paid
1,2,High,Medium,Medium,Medium,Paid
2,3,Medium,Medium,Medium,High,Not_Paid
3,4,Low,High,High,Low,Paid
4,5,Medium,Medium,Low,Low,Paid
5,6,Medium,Medium,High,Medium,Paid
6,7,High,Low,Medium,Low,Paid
7,8,Medium,Low,Medium,Low,Paid
8,9,Medium,Medium,Medium,Medium,Paid
9,10,Medium,High,Low,Low,Paid


In [12]:
# Class distribution queried from the database (should be 84.0% Paid / 16.0% Not_Paid)
pd.read_sql("""
SELECT not_fully_paid, COUNT(*) AS n,
       ROUND(100.0*COUNT(*)/(SELECT COUNT(*) FROM loan_full), 1) AS pct
FROM loan_full GROUP BY not_fully_paid""", conn)

,not_fully_paid,n,pct
0,0,8045,84.0
1,1,1533,16.0
